# 04 - Model Evaluation (NumPy)
RMSE, MAE, F1, SSI, Wasserstein Distance

**Inputs (from Notebook 03):** `convlstm_weights.npz`, `X_test.npy`, `y_test.npy`

**Outputs:** `predictions.npy`, `evaluation_results.csv`

> This notebook uses the same pure-NumPy ConvLSTM model defined in
> `03_buildntrain_numpy.ipynb`. No PyTorch or TensorFlow required.

In [1]:
# CELL 1 — pip install (scikit-learn + scipy only; no torch)
!pip install -q scikit-learn scipy

In [2]:
# CELL 2 — imports + CONFIG
# Identical CONFIG to Notebook 03 so all file paths resolve the same way.

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, f1_score
from scipy.stats import wasserstein_distance
import json, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CONFIG = {
    # Spatial
    'bbox_regional': {'lat_min': 0,  'lat_max': 30,  'lon_min': 110, 'lon_max': 140},
    'bbox_model':    {'lat_min': 10, 'lat_max': 20,  'lon_min': 114, 'lon_max': 120},
    # Temporal
    'date_full':  {'start': '2014-01-01', 'end': '2024-12-31'},
    'date_model': {'start': '2019-01-01', 'end': '2024-12-31'},
    # Paths
    'data_dir': '/content/drive/MyDrive/fishing_project/',
    'files': {
        'physics_w_nc':    'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779636039565.nc',
        'physics_ht_nc':   'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779635380319.nc',
        'bgc_src_nc':      'cmems_mod_glo_bgc_my_0.25deg_P1M-m_1779635372583.nc',
        'physics_nc':      'physics_raw_region.nc',
        'bgc_nc':          'bgc_raw_region.nc',
        'ais_parquet':     'ais_raw_region.parquet',
        'ais_csv_gz':      'ais_raw_region.csv.gz',
        'ais_gridded_nc':  'ais_fishing_effort_gridded.nc',
        'preprocessed_nc': 'preprocessed_features.nc',
        # NumPy model weights (from 03_buildntrain_numpy)
        'weights_npz':     'convlstm_weights.npz',
        # Placeholder files (text files written by 03 for dashboard compat)
        'model_keras':     'convlstm_model.keras',
        'best_model':      'best_model.keras',
        # Shared outputs — identical names across pipeline
        'X_test_npy':      'X_test.npy',
        'y_test_npy':      'y_test.npy',
        'history_json':    'training_history.json',
        'summary_json':    'data_summary.json',
        'predictions_npy': 'predictions.npy',
        'eval_csv':        'evaluation_results.csv',
    },
    'ais_use_cols':          ['date', 'cell_ll_lat', 'cell_ll_lon', 'fishing_hours'],
    'physics_surface_depth': 0.49,
    'bgc_depth_range':       (0.51, 5.14),
    'norm_method':           'minmax',
    'resample_freq':         '1ME',
    'seq_len':    3,
    'pred_len':   1,
    'n_channels': 7,
    'train_frac': 0.70,
    'val_frac':   0.15,
    'epochs':     50,
    'batch_size': 8,
    'patience':   10,
    'f1_threshold': 0.15,
}

DATA_DIR = CONFIG['data_dir']
f        = CONFIG['files']

print(f"NumPy  : {np.__version__}")
print("No torch / tensorflow — pure NumPy inference.")

In [3]:
# CELL 3 — Re-define the NumPy ConvLSTM model (copy from 03_buildntrain_numpy)
#
# We copy only the forward-pass code needed for inference.
# No backward / optimizer code needed here.

N_CHANNELS = CONFIG['n_channels']
SEQ_LEN    = CONFIG['seq_len']

FILTERS1 = 16
FILTERS2 = 8


# ── im2col ────────────────────────────────────────────────────────────────────

def im2col(x, kH, kW, pad):
    B, H, W, C = x.shape
    if pad > 0:
        x_padded = np.pad(x, ((0,0),(pad,pad),(pad,pad),(0,0)), mode='constant')
    else:
        x_padded = x
    Hp, Wp = x_padded.shape[1], x_padded.shape[2]
    H_out  = Hp - kH + 1
    W_out  = Wp - kW + 1
    sb, sh, sw, sc = x_padded.strides
    patches = np.lib.stride_tricks.as_strided(
        x_padded,
        shape  = (B, H_out, W_out, kH, kW, C),
        strides= (sb, sh, sw, sh, sw, sc),
    )
    col = patches.reshape(B * H_out * W_out, kH * kW * C)
    return col, x_padded, H_out, W_out


def conv_forward(x, W, b, kH, kW, pad):
    """2D convolution forward (channel-last). Returns (out, cache)."""
    B, H, W_in, C_in = x.shape
    col, x_padded, H_out, W_out = im2col(x, kH, kW, pad)
    out_flat = col @ W + b
    out      = out_flat.reshape(B, H_out, W_out, -1)
    cache    = (col, x_padded, x.shape, H_out, W_out)
    return out, cache


# ── Activations ───────────────────────────────────────────────────────────────

def sigmoid(x):
    pos = x >= 0
    out = np.where(pos,
                   1.0 / (1.0 + np.exp(-np.abs(x))),
                   np.exp(x) / (1.0 + np.exp(x)))
    return out.astype(x.dtype)

def tanh_act(x):
    return np.tanh(x).astype(x.dtype)


# ── ConvLSTMCell (inference-only) ─────────────────────────────────────────────

class ConvLSTMCell:
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        self.F   = hidden_channels
        self.kH  = kernel_size
        self.kW  = kernel_size
        self.pad = kernel_size // 2
        col_x = kernel_size * kernel_size * in_channels
        col_h = kernel_size * kernel_size * hidden_channels
        self.Wx = np.zeros((col_x, 4 * hidden_channels), dtype=np.float32)
        self.Wh = np.zeros((col_h, 4 * hidden_channels), dtype=np.float32)
        self.b  = np.zeros(4 * hidden_channels, dtype=np.float32)

    def forward(self, x_t, h_prev, c_prev):
        gates_x, _ = conv_forward(x_t,    self.Wx, np.zeros(4*self.F, dtype=np.float32),
                                   self.kH, self.kW, self.pad)
        gates_h, _ = conv_forward(h_prev, self.Wh, np.zeros(4*self.F, dtype=np.float32),
                                   self.kH, self.kW, self.pad)
        gates = gates_x + gates_h + self.b
        F = self.F
        i = sigmoid(gates[..., :F])
        f = sigmoid(gates[..., F:2*F])
        g = tanh_act(gates[..., 2*F:3*F])
        o = sigmoid(gates[..., 3*F:])
        c_t = f * c_prev + i * g
        h_t = o * tanh_act(c_t)
        return h_t, c_t


class ConvLSTMLayer:
    def __init__(self, in_channels, hidden_channels, kernel_size=3,
                 return_sequences=True):
        self.cell             = ConvLSTMCell(in_channels, hidden_channels, kernel_size)
        self.F                = hidden_channels
        self.return_sequences = return_sequences

    def forward(self, x_seq):
        """x_seq : (B, T, H, W, C_in)"""
        B, T, H, W, C = x_seq.shape
        h = np.zeros((B, H, W, self.F), dtype=np.float32)
        c = np.zeros((B, H, W, self.F), dtype=np.float32)
        outputs = []
        for t in range(T):
            h, c = self.cell.forward(x_seq[:, t], h, c)
            outputs.append(h)
        if self.return_sequences:
            return np.stack(outputs, axis=1)
        return outputs[-1]


class BatchNorm:
    """Inference-mode only: always uses running stats."""
    def __init__(self, num_channels, eps=1e-5):
        self.eps    = eps
        self.gamma  = np.ones(num_channels,  dtype=np.float32)
        self.beta   = np.zeros(num_channels, dtype=np.float32)
        self.r_mean = np.zeros(num_channels, dtype=np.float32)
        self.r_var  = np.ones(num_channels,  dtype=np.float32)

    def forward(self, x):
        """x : (..., C) — channel is last dim. Uses running stats."""
        x_hat = (x - self.r_mean) / np.sqrt(self.r_var + self.eps)
        return self.gamma * x_hat + self.beta


class ConvLSTMModel:
    def __init__(self, n_channels=7, filters1=16, filters2=8,
                 kernel_size=3):
        self.F1     = filters1
        self.F2     = filters2
        self.layer1 = ConvLSTMLayer(n_channels, filters1, kernel_size,
                                    return_sequences=True)
        self.layer2 = ConvLSTMLayer(filters1,   filters2, kernel_size,
                                    return_sequences=False)
        self.bn1    = BatchNorm(filters1)
        self.bn2    = BatchNorm(filters2)
        lim         = np.sqrt(6.0 / (filters2 + 1))
        self.W_out  = np.random.uniform(-lim, lim, (filters2, 1)).astype(np.float32)
        self.b_out  = np.zeros(1, dtype=np.float32)

    def forward(self, x):
        """Inference forward. x : (B, T, H, W, C). Returns (B, H, W, 1) in (0,1)."""
        # Block 1
        out1    = self.layer1.forward(x)          # (B, T, H, W, F1)
        out1_bn = self.bn1.forward(out1)          # BatchNorm (inference)
        # No dropout at inference

        # Block 2
        out2    = self.layer2.forward(out1_bn)    # (B, H, W, F2)
        out2_bn = self.bn2.forward(out2)

        # Output head
        logit = out2_bn @ self.W_out + self.b_out  # (B, H, W, 1)
        return sigmoid(logit)

    def count_params(self):
        total = 0
        for arr in [
            self.layer1.cell.Wx, self.layer1.cell.Wh, self.layer1.cell.b,
            self.bn1.gamma, self.bn1.beta,
            self.layer2.cell.Wx, self.layer2.cell.Wh, self.layer2.cell.b,
            self.bn2.gamma, self.bn2.beta,
            self.W_out, self.b_out,
        ]:
            total += arr.size
        return total


def load_weights(model, path):
    """Load weights from .npz file saved by 03_buildntrain_numpy."""
    w = np.load(path)
    model.layer1.cell.Wx[:] = w['l1_Wx']
    model.layer1.cell.Wh[:] = w['l1_Wh']
    model.layer1.cell.b[:]  = w['l1_b']
    model.bn1.gamma[:]      = w['bn1_gamma']
    model.bn1.beta[:]       = w['bn1_beta']
    model.bn1.r_mean[:]     = w['bn1_rmean']
    model.bn1.r_var[:]      = w['bn1_rvar']
    model.layer2.cell.Wx[:] = w['l2_Wx']
    model.layer2.cell.Wh[:] = w['l2_Wh']
    model.layer2.cell.b[:]  = w['l2_b']
    model.bn2.gamma[:]      = w['bn2_gamma']
    model.bn2.beta[:]       = w['bn2_beta']
    model.bn2.r_mean[:]     = w['bn2_rmean']
    model.bn2.r_var[:]      = w['bn2_rvar']
    model.W_out[:]          = w['W_out']
    model.b_out[:]          = w['b_out']


# -- Instantiate and load weights -------------------------------------------
model      = ConvLSTMModel(n_channels=N_CHANNELS, filters1=FILTERS1, filters2=FILTERS2)
weights_path = DATA_DIR + f['weights_npz']

if not os.path.exists(weights_path):
    raise FileNotFoundError(
        f"No NumPy weights found at {weights_path}.\n"
        "Run 03_buildntrain_numpy.ipynb first."
    )

load_weights(model, weights_path)
print(f"Loaded weights from: {weights_path}")
print(f"Parameters: {model.count_params():,}")

# -- Load test arrays (channel-last layout from notebook 03) -----------------
X_test = np.load(DATA_DIR + f['X_test_npy'])   # (N, T, H, W, C) channel-last
y_test = np.load(DATA_DIR + f['y_test_npy'])   # (N, H, W, 1)

print(f"X_test shape  : {X_test.shape}")
print(f"y_test shape  : {y_test.shape}")

In [4]:
# CELL 4 — generate predictions and save to predictions.npy
#
# Layout: channel-last throughout (N, T, H, W, C) -> model -> (N, H, W, 1)
# No transpose needed — NumPy model uses the same channel-last convention
# as Notebook 03, so X_test can be fed directly.

X_clean    = np.nan_to_num(X_test, nan=0.0)  # (N, T, H, W, C)
preds_list = []
batch_size = CONFIG['batch_size']

for start in range(0, len(X_clean), batch_size):
    xb = X_clean[start : start + batch_size]   # (B, T, H, W, C) — channel-last
    pb = model.forward(xb)                      # (B, H, W, 1)
    preds_list.append(pb)

preds = np.concatenate(preds_list, axis=0)     # (N, H, W, 1)

print(f'Predictions shape: {preds.shape}')     # Expected: (N, 41, 25, 1)

np.save(DATA_DIR + f['predictions_npy'], preds)
print(f'Saved predictions -> {f["predictions_npy"]}')

# Flatten for global scalar metrics, cleaning any residual NaNs
y_flat = np.nan_to_num(y_test.flatten(), nan=0.0)
p_flat = np.nan_to_num(preds.flatten(), nan=0.0)
print(f'NaN in y_flat : {np.isnan(y_flat).sum()}')
print(f'NaN in p_flat : {np.isnan(p_flat).sum()}')

In [5]:
# CELL 5 — RMSE and MAE (global scalars over all test pixels and months)
rmse = float(np.sqrt(mean_squared_error(y_flat, p_flat)))
mae  = float(mean_absolute_error(y_flat, p_flat))
print(f'RMSE : {rmse:.6f}')
print(f'MAE  : {mae:.6f}')

In [6]:
# CELL 6 — F1 score with threshold sweep (finds optimal threshold)
thresholds_to_test = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
f1_scores = []

for thresh in thresholds_to_test:
    y_binary    = (y_flat > thresh).astype(int)
    pred_binary = (p_flat > thresh).astype(int)
    f1 = f1_score(y_binary, pred_binary, zero_division=0)
    f1_scores.append(f1)
    print(f'Threshold {thresh:.2f} -> F1 = {f1:.4f}')

best_idx       = int(np.argmax(f1_scores))
best_threshold = thresholds_to_test[best_idx]
best_f1        = f1_scores[best_idx]
print(f'\n** Best F1: {best_f1:.4f} at threshold {best_threshold:.2f} **')

config_threshold = CONFIG['f1_threshold']
y_binary_cfg    = (y_flat > config_threshold).astype(int)
pred_binary_cfg = (p_flat > config_threshold).astype(int)
f1_config = float(f1_score(y_binary_cfg, pred_binary_cfg, zero_division=0))
print(f'F1 at CONFIG threshold ({config_threshold}): {f1_config:.4f}')

f1 = best_f1

In [7]:
# CELL 7 — SSI (Structural Similarity Index) — per-sample, then averaged
def calculate_ssi(obs, pred):
    C1, C2 = 0.01**2, 0.03**2
    mu_obs,    mu_pred    = obs.mean(),  pred.mean()
    sigma_obs, sigma_pred = obs.std(),   pred.std()
    sigma_cross = np.mean((obs - mu_obs) * (pred - mu_pred))
    luminance = (2*mu_obs*mu_pred + C1) / (mu_obs**2 + mu_pred**2 + C1)
    contrast  = (2*sigma_obs*sigma_pred + C2) / (sigma_obs**2 + sigma_pred**2 + C2)
    structure = (sigma_cross + C2/2) / (sigma_obs*sigma_pred + C2/2)
    return luminance * contrast * structure

ssi_scores = []
for i in range(len(y_test)):
    yi = y_test[i].flatten()
    pi = preds[i].flatten()
    m  = ~(np.isnan(yi) | np.isnan(pi))
    ssi_scores.append(calculate_ssi(yi[m], pi[m]))

ssi_mean = float(np.mean(ssi_scores))
ssi_std  = float(np.std(ssi_scores))
print(f'SSI : {ssi_mean:.4f} +/- {ssi_std:.4f}')

In [8]:
# CELL 8 — Wasserstein Distance — per-sample distribution comparison
wd_scores = []
for i in range(len(y_test)):
    obs_flat  = y_test[i].flatten()
    pred_flat = preds[i].flatten()
    m = ~(np.isnan(obs_flat) | np.isnan(pred_flat))
    obs_flat, pred_flat = obs_flat[m], pred_flat[m]
    obs_sum  = obs_flat.sum()
    pred_sum = pred_flat.sum()
    if obs_sum == 0 or pred_sum == 0:
        wd_scores.append(np.nan)
        continue
    wd = wasserstein_distance(
        np.arange(len(obs_flat)),
        np.arange(len(pred_flat)),
        obs_flat / obs_sum,
        pred_flat / pred_sum,
    )
    wd_scores.append(wd)

wd_valid = [s for s in wd_scores if not np.isnan(s)]
wd_mean  = float(np.mean(wd_valid))
wd_std   = float(np.std(wd_valid))
print(f'Wasserstein Distance : {wd_mean:.4f} +/- {wd_std:.4f}')

In [9]:
# CELL 9 — compile results table and save to evaluation_results.csv
results = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'F1_best', 'F1_config', 'SSI', 'Wasserstein'],
    'Value': [
        f'{rmse:.3e}',
        f'{mae:.3e}',
        f'{best_f1:.4f}',
        f'{f1_config:.4f}',
        f'{ssi_mean:.4f}',
        f'{wd_mean:.4f}',
    ],
    'Std Dev': [
        'N/A', 'N/A',
        f'(thresh={best_threshold})',
        f'(thresh={config_threshold})',
        f'{ssi_std:.4f}',
        f'{wd_std:.4f}',
    ],
})

print('=' * 50)
print('EVALUATION RESULTS')
print('=' * 50)
print(results.to_string(index=False))

results.to_csv(DATA_DIR + f['eval_csv'], index=False)
print(f'\nSaved evaluation results -> {f["eval_csv"]}')
print('Notebook 04 complete. Run 05_visualization.ipynb next.')